In [7]:
import os
import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
from glob import glob
import numpy as np
from spectral.io import envi

In [2]:
# file paths
home = '/store/carroll/sbgplants/'
ref = os.path.join(home, 'schema')
raw = os.path.join(home, 'data', 'raw')
flights = '/store/carroll/col/2018/raw/rmbl/' # still only 4 / 7 days, will rerun once have everything

out_folder = os.path.join(home, 'data', 'out_csv')

table = 'extracted_spectra'

In [3]:
# load and view relevant schema and dtype for the table
schema = pd.read_csv(os.path.join(ref, 'sbgplants-schema.csv'))
data_types = pd.read_csv(os.path.join(ref, 'data-types.csv'))

# fix typos
schema['column_name'] = schema['column_name'].replace('wavelenght_center', 'wavelength_center')

schema = schema[schema.table_name==table]
schema

,table_name,column_name,data_type
8,extracted_spectra,band_number,integer
9,extracted_spectra,radiance,double precision
10,extracted_spectra,reflectance,double precision
11,extracted_spectra,uncertainty_ref,double precision
12,extracted_spectra,extract_id,uuid
13,extracted_spectra,pixel_id,uuid


In [4]:
# load relevant output tables

pixel = pd.read_csv(os.path.join(out_folder, 'pixel.csv'))
flightlines = pixel.flightline_id.unique()

In [12]:
# extract rdn spectra

fps = [x for x in glob(os.path.join(flights, '*/*rdn_ort.hdr')) if any(xx in x for xx in flightlines)]
n_bands = len(envi.read_envi_header(fps[0])['wavelength'])
id_cols = pixel.columns
val_cols = [x for x in range(n_bands)]

out = []

for fp in fps:
    # filter px to flightline
    flight = fp.split('/')[-1].removesuffix('_rdn_ort.hdr')
    tmp = pixel[pixel['flightline_id']==flight].copy()
    # extract rdn
    rdn = envi.open(fp).open_memmap()
    r = tmp['glt_row']; c=tmp['glt_column']
    vals = rdn[r, c, :]
    # format df
    tmp = pd.concat([tmp, pd.DataFrame(vals, index=tmp.index, columns=val_cols)], axis=1)
    tmp = tmp.melt(id_vars=id_cols, value_vars=val_cols, var_name='band_number', value_name='radiance')
    out.append(tmp)

df = pd.concat(out)

In [13]:
df

,plot_name,glt_row,glt_column,flightline_id,pixel_id,scene_id,band_number,radiance
0,056-ER18,287,51,NIS01_20180612_175319,px0,NaN,0,2.040740
1,056-ER18,287,52,NIS01_20180612_175319,px1,NaN,0,1.962031
2,056-ER18,288,51,NIS01_20180612_175319,px2,NaN,0,1.906291
3,056-ER18,288,52,NIS01_20180612_175319,px3,NaN,0,2.055831
4,056-ER18,287,51,NIS01_20180612_175319,px0,NaN,1,1.702304
...,...,...,...,...,...,...,...,...
46429,205-ER18,16984,135,NIS01_20180620_173209,px5471,NaN,425,0.010496
46430,197-ER18,16992,143,NIS01_20180620_173209,px5472,NaN,425,0.010736
46431,197-ER18,16992,144,NIS01_20180620_173209,px5473,NaN,425,0.011682
46432,204-ER18,17004,118,NIS01_20180620_173209,px5474,NaN,425,0.015537


In [14]:
# prepare & populate out table
out_table = df

out_table['reflectance'] = pd.NA
out_table['uncertainty_ref'] = pd.NA
out_table['extract_id'] = pd.NA

# reorder columns
out_table = out_table[schema.column_name]

out_table

,band_number,radiance,reflectance,uncertainty_ref,extract_id,pixel_id
0,0,2.040740,<NA>,<NA>,<NA>,px0
1,0,1.962031,<NA>,<NA>,<NA>,px1
2,0,1.906291,<NA>,<NA>,<NA>,px2
3,0,2.055831,<NA>,<NA>,<NA>,px3
4,1,1.702304,<NA>,<NA>,<NA>,px0
...,...,...,...,...,...,...
46429,425,0.010496,<NA>,<NA>,<NA>,px5471
46430,425,0.010736,<NA>,<NA>,<NA>,px5472
46431,425,0.011682,<NA>,<NA>,<NA>,px5473
46432,425,0.015537,<NA>,<NA>,<NA>,px5474


In [15]:
# check, update final column data types

print(out_table.dtypes)

out_table.loc[:,'band_number'] = out_table['band_number'].astype(int)

out_table.dtypes

band_number         object
radiance           float32
reflectance         object
uncertainty_ref     object
extract_id          object
pixel_id            object
dtype: object


band_number         object
radiance           float32
reflectance         object
uncertainty_ref     object
extract_id          object
pixel_id            object
dtype: object

In [16]:
# export table
fp_out = os.path.join(out_folder, f'{table}.csv')
out_table.to_csv(fp_out, index=False)